# Goemans–Williamson MaxCut 小规模验证

这本 notebook 验证 standalone Goemans–Williamson 路线：CVXPY SDP relaxation、PSD factorization 和 seeded random-hyperplane rounding。问题与算法全部定义在脚本中；notebook 只负责调用、对照与断言。

> GW 是 approximation algorithm。即使命中真实最优解，正常结果仍应是 `feasible`，不能仅凭 SDP 与有限次 rounding 声明 `optimal`。

## 1. 环境与导入

需要项目 `.venv` 中已安装 `cvxpy`，并至少有 CLARABEL 或 SCS。

In [ ]:
import json
import sys
from pathlib import Path

import cvxpy as cp


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, GoemansWilliamsonQuboSolver
from problem.reproductions import build_qrbnbr_s1_instance
from tests.oracles import enumerate_qubo, public_json_number

print("Python:", sys.executable)
print("CVXPY:", cp.__version__)
print("Installed solvers:", cp.installed_solvers())

## 2. S1 风格 MaxCut 与独立最优值

同一确定性问题同时交给独立 `Fraction` oracle、Exact solver 和 GW。

In [ ]:
problem = build_qrbnbr_s1_instance(
    variable_count=10,
    edge_probability=0.45,
    seed=2025,
)
validate_qubo(problem)

oracle_rows = enumerate_qubo(problem)
oracle_energy = public_json_number(oracle_rows[0]["energy_exact"])
exact_result = ExactQuboSolver().solve(problem)

assert exact_result["status"] == "optimal"
assert exact_result["best_energy"] == oracle_energy

print("Problem:", problem["problem_id"])
print("Variables:", problem["num_variables"])
print("Oracle optimum energy:", oracle_energy)

## 3. CLARABEL SDP 与 seeded hyperplanes

固定 seed 只保证 rounding 路径可复现；它不把 approximation result 变成 exact proof。

In [ ]:
config = {
    "rounds": 128,
    "seed": 7,
    "sdp_solver": "CLARABEL",
    "sdp_tolerance": 1e-7,
}
gw_result = GoemansWilliamsonQuboSolver().solve(problem, config)
gw_repeat = GoemansWilliamsonQuboSolver().solve(problem, config)

validate_qubo_result(problem, gw_result)
assert gw_result["status"] == "feasible"
assert gw_result["best_sample"] == gw_repeat["best_sample"]
assert gw_result["best_energy"] == gw_repeat["best_energy"]
assert gw_result["best_energy"] >= oracle_energy
assert gw_result["metrics"]["sdp_relaxation_value"] + 1e-5 >= gw_result["metrics"]["cut_value"]
json.dumps(gw_result, allow_nan=False)

print("GW sample:", gw_result["best_sample"])
print("GW energy:", gw_result["best_energy"])
print("SDP relaxation:", gw_result["metrics"]["sdp_relaxation_value"])
print("Cut / SDP:", gw_result["metrics"].get("cut_to_sdp_ratio"))

## 4. 可替换 SDP backend

若当前环境安装了 SCS，再运行同一模型并检查两种 backend 的候选与 relaxation 数值。

In [ ]:
scs_result = None
if "SCS" in cp.installed_solvers():
    scs_result = GoemansWilliamsonQuboSolver().solve(
        problem,
        {**config, "sdp_solver": "SCS"},
    )
    validate_qubo_result(problem, scs_result)
    assert scs_result["status"] == "feasible"
    assert scs_result["best_energy"] >= oracle_energy
    print("SCS energy:", scs_result["best_energy"])
    print("SCS relaxation:", scs_result["metrics"]["sdp_relaxation_value"])
else:
    print("SCS is not installed; CLARABEL validation remains complete.")

## 结论

- canonical QUBO 能量、GW cut 和 SDP relaxation 的方向一致；
- fixed seed 重复 GW rounding 的算法结果；
- 命中 oracle optimum 仍保持 `feasible`；
- SDP backend、状态、迭代数与时间以 diagnostics 报告，不作为 exact promotion。